# Khám phá dữ liệu PM2.5 tại Hoàn Kiếm

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "axes.titleweight": "bold", "axes.spines.top": False, "axes.spines.right": False})

df = pd.read_csv(r"datadone_dirty.csv", low_memory=False)
df["Local Time"] = pd.to_datetime(df["Local Time"], format="mixed", errors="coerce")
df["UTC Time"] = pd.to_datetime(df["UTC Time"], format="mixed", errors="coerce")

pollutant_cols = ["CO", "NO2", "O3", "PM10", "PM25", "SO2"]
weather_cols = ["Clouds", "Precipitation", "Pressure", "Relative Humidity", "Temperature", "UV Index", "Wind Speed"]
numeric_cols = ["AQI", *pollutant_cols, *weather_cols]
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")


FileNotFoundError: [Errno 2] No such file or directory: 'datadone_dirty.csv'

## 1. Dữ liệu được tạo ra như thế nào, chứa những thông tin gì và có phạm vi sử dụng ra sao?

In [ ]:
print(f"Kích thước dữ liệu raw: {df.shape[0]:,} dòng x {df.shape[1]} cột")
print("Nguồn:", df[['City','Country Code','Timezone']].drop_duplicates().to_dict('records'))
print("Phạm vi thời gian:", df['Local Time'].min(), "->", df['Local Time'].max())
display(df.head(3))


In [ ]:
data_dictionary = pd.DataFrame([
    ("Local Time, UTC Time", "datetime", "-", "Thời gian địa phương và quốc tế"),
    ("AQI", "int", "Chỉ số", "Chỉ số chất lượng không khí"),
    ("PM25", "float", "µg/m³", "Nồng độ bụi PM2.5 - biến mục tiêu"),
    ("PM10, CO, NO2, SO2, O3", "float", "µg/m³", "Các chất ô nhiễm khác"),
    ("Temperature", "float", "°C", "Nhiệt độ không khí"),
    ("Relative Humidity, Clouds", "int", "%", "Độ ẩm tương đối, độ che phủ mây"),
    ("Pressure", "float", "mb/hPa", "Áp suất không khí"),
    ("Precipitation", "float", "mm", "Lượng mưa tích lũy theo giờ"),
    ("Wind Speed", "float", "m/s", "Tốc độ gió"),
    ("UV Index", "float", "Chỉ số", "Chỉ số tia cực tím"),
], columns=["Thuộc tính", "Kiểu dữ liệu", "Đơn vị", "Ý nghĩa"])
display(data_dictionary)


## 2. Có căn cứ nào từ mục tiêu nghiên cứu và đặc điểm dữ liệu để chọn PM2.5 làm biến cần dự báo?

In [ ]:
corr_aqi = df["AQI"].corr(df["PM25"], method="spearman")
ac1 = df["PM25"].autocorr(1)
ac24 = df["PM25"].autocorr(24)
print(f"Tương quan Spearman AQI - PM2.5: {corr_aqi:.3f}")
print(f"Tự tương quan PM2.5 (lag 1h): {ac1:.3f}")
print(f"Tự tương quan PM2.5 (lag 24h): {ac24:.3f}")


## 3. Nồng độ PM2.5 có phân bố và mức độ biến động như thế nào? Giá trị cao/khác biệt xuất hiện khi nào?

In [ ]:
display(df["PM25"].describe().to_frame().T)
print("Skewness:", round(df["PM25"].skew(), 2))
display(df.nlargest(10, "PM25")[["Local Time", "PM25", "AQI"]])


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
sns.histplot(df["PM25"].dropna(), bins=50, ax=ax[0])
ax[0].set_title("Phân bố PM2.5")
sns.boxplot(y=df["PM25"], ax=ax[1])
ax[1].set_title("Boxplot PM2.5")
plt.tight_layout()
plt.show()


## 4. PM2.5 thay đổi thế nào theo giờ, tháng, mùa, năm? Có tính chu kỳ theo thời gian?

In [ ]:
by_hour = df.groupby(df["Local Time"].dt.hour)["PM25"].mean()
by_month = df.groupby(df["Local Time"].dt.month)["PM25"].mean()
by_year = df.groupby(df["Local Time"].dt.year)["PM25"].mean()

fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
by_hour.plot(ax=ax[0], marker="o", title="Theo giờ trong ngày")
by_month.plot(ax=ax[1], marker="o", color="orange", title="Theo tháng")
by_year.plot(ax=ax[2], kind="bar", color="seagreen", title="Theo năm")
plt.tight_layout()
plt.show()

for lag, label in [(24, "24h (1 ngày)"), (168, "168h (1 tuần)")]:
    print(f"Tự tương quan lag {label}: {df['PM25'].autocorr(lag):.3f}")


## 5. PM2.5 có mối liên hệ thế nào với các chất ô nhiễm khác và yếu tố khí tượng? Có khác nhau giữa các giai đoạn không?

In [ ]:
other_vars = [c for c in pollutant_cols + weather_cols if c != "PM25"]
corr_overall = df[["PM25"] + other_vars].corr(method="spearman")["PM25"].drop("PM25").sort_values()
display(corr_overall.to_frame("Tương quan Spearman"))

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(df[["PM25"] + other_vars].corr(method="spearman"), annot=True, fmt=".2f", cmap="coolwarm", ax=ax)
plt.title("Ma trận tương quan (Spearman)")
plt.show()


In [ ]:
df["Year"] = df["Local Time"].dt.year
corr_by_year = df.groupby("Year")[["PM25", "PM10", "Temperature", "Pressure"]].corr(method="spearman")["PM25"].unstack()
display(corr_by_year.round(2))


## 6. Dữ liệu có vấn đề gì về giá trị thiếu, trùng lặp, tính hợp lệ và tính liên tục theo thời gian? Ảnh hưởng thế nào?

In [ ]:
missing_table = df[pollutant_cols + weather_cols].isna().sum()
missing_table = missing_table[missing_table > 0].to_frame("Số giá trị thiếu")
missing_table["Tỷ lệ (%)"] = (missing_table["Số giá trị thiếu"] / len(df) * 100).round(2)
display(missing_table)


In [ ]:
print("Dòng trùng hoàn toàn:", df.duplicated().sum())
print("Local Time bị lặp:", df["Local Time"].duplicated().sum())
display(df.loc[df["Local Time"].duplicated(keep=False), ["Local Time", "PM25", "AQI"]].sort_values("Local Time"))


In [ ]:
expected = pd.date_range(df["Local Time"].min(), df["Local Time"].max(), freq="h")
missing_hours = expected.difference(df["Local Time"].dropna())
print("Số giờ bị thiếu trong lưới thời gian:", len(missing_hours))
display(pd.DataFrame({"Timestamp bị thiếu": missing_hours}))


In [ ]:
validity = pd.Series({
    "Timestamp không parse được": int(df["Local Time"].isna().sum()),
    "Nồng độ chất ô nhiễm âm": int((df[pollutant_cols] < 0).sum().sum()),
    "Clouds/Humidity ngoài [0,100]": int(((df["Clouds"] < 0) | (df["Clouds"] > 100) | (df["Relative Humidity"] < 0) | (df["Relative Humidity"] > 100)).sum()),
})
display(validity.to_frame("Số dòng vi phạm"))


## 7. Sau các kiểm tra trên, bộ dữ liệu có phù hợp để tiếp tục sử dụng cho mục tiêu dự báo PM2.5 hay không?